#Configuration

In [0]:
STORAGE_ACCOUNT = "cmsmedicaredatastorage"
CATALOG         = "cms_medicare_databricks_pipeline"
SOURCE_TABLE    = f"{CATALOG}.bronze.readmissions_raw"
TARGET_TABLE    = f"{CATALOG}.silver.readmissions"

#Silver Readmissions in California

In [0]:
%sql
-- Silver Readmissions in California
CREATE OR REPLACE TABLE cms_medicare_databricks_pipeline.silver.readmissions AS

SELECT
    -- Hospital identity
    facility_id,
    facility_name,
    state,
    measure_name, --Condition Codes
    -- Human-readable condition name
    CASE measure_name
        WHEN 'READM-30-AMI-HRRP'      THEN 'Acute Myocardial Infarction (Heart Attack)'
        WHEN 'READM-30-CABG-HRRP'     THEN 'Coronary Artery Bypass Graft'
        WHEN 'READM-30-COPD-HRRP'     THEN 'Chronic Obstructive Pulmonary Disease'
        WHEN 'READM-30-HF-HRRP'       THEN 'Heart Failure'
        WHEN 'READM-30-HIP-KNEE-HRRP' THEN 'Hip & Knee Replacement'
        WHEN 'READM-30-PN-HRRP'       THEN 'Pneumonia'
        ELSE 'Unknown'
    END AS condition_name,
    -- Volume
    TRY_CAST(number_of_readmissions AS INTEGER) AS number_of_readmissions,
    TRY_CAST(number_of_discharges AS INTEGER)   AS number_of_discharges,

    -- Readmission metrics
    -- TRY_CAST handles ALL non-numeric values gracefully:
    -- 'N/A', 'Too Few to Report', empty strings -> all become NULL
    TRY_CAST(excess_readmission_ratio AS DECIMAL(10,4))   AS excess_readmission_ratio,
    -- KEY METRIC: ratio of predicted to expected readmissions
    -- > 1.0 = more readmissions than expected = poor performance
    -- < 1.0 = fewer readmissions than expected = good performance
    -- = 1.0 = exactly as expected
    TRY_CAST(predicted_readmission_rate AS DECIMAL(10,4)) AS predicted_readmission_rate,
    TRY_CAST(expected_readmission_rate AS DECIMAL(10,4))  AS expected_readmission_rate,

    TO_DATE(start_date, 'MM/dd/yyyy') AS start_date,
    TO_DATE(end_date, 'MM/dd/yyyy')   AS end_date,

    footnote,
    ingestion_timestamp,
    source_file
FROM
    cms_medicare_databricks_pipeline.bronze.readmissions_raw
WHERE
    state = 'CA'

In [0]:
%sql
-- Dedup pass: keep only the most recent row per hospital-condition pair
CREATE OR REPLACE TABLE cms_medicare_databricks_pipeline.silver.readmissions AS
SELECT * EXCEPT (rn)
FROM (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY facility_id, measure_name
            ORDER BY ingestion_timestamp DESC
        ) AS rn
    FROM cms_medicare_databricks_pipeline.silver.readmissions
)
WHERE rn = 1

#Data Quality Check

In [0]:
%sql
-- Check how many rows have N/A suppressed values
SELECT
    condition_name,
    COUNT(*)                                         AS total_hospitals,
    SUM(CASE WHEN excess_readmission_ratio IS NULL
        THEN 1 ELSE 0 END)                           AS suppressed_rows,
    -- Suppressed = CMS withheld data (low volume or other reason)
    ROUND(AVG(excess_readmission_ratio), 4)          AS avg_excess_ratio,
    ROUND(MIN(excess_readmission_ratio), 4)          AS min_excess_ratio,
    ROUND(MAX(excess_readmission_ratio), 4)          AS max_excess_ratio
FROM cms_medicare_databricks_pipeline.silver.readmissions
GROUP BY condition_name
ORDER BY avg_excess_ratio DESC

#Verify

In [0]:
%sql
SELECT
    COUNT(*)                    AS total_rows,
    COUNT(DISTINCT facility_id) AS unique_hospitals,
    COUNT(DISTINCT measure_name) AS unique_conditions,
    COUNT(DISTINCT state)       AS states
FROM cms_medicare_databricks_pipeline.silver.readmissions